In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from ultralytics import YOLO

# ============================================================
# SETTINGS
# ============================================================

VIDEO_PATH = r"\\192.168.1.10\shear all\test_vedio\VID00051.AVI"
OUTPUT_PATH = r"C:\Users\PC-LAB1\Desktop\shok\notebooks\Day_08\drone_tracking_result.mp4"

MODEL_PATH = "yolov8n.pt"

CONFIDENCE = 0.25

# COCO classes
# 0  = person
# 2  = car
# 15 = cat
TARGET_CLASSES = {
    0: "Person",
    2: "Car",
    15: "Cat"
}

TRAIL_LENGTH = 30

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

print("Model loaded.")

# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video:\n{VIDEO_PATH}"
    )

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

duration = total_frames / fps

print()
print("VIDEO INFORMATION")
print("------------------")
print(f"Resolution : {width} x {height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")
print(f"Duration   : {duration:.2f} seconds")

# ============================================================
# OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

# ============================================================
# TRACKING DATA
# ============================================================

previous_positions = {}

trails = defaultdict(
    lambda: deque(maxlen=TRAIL_LENGTH)
)

speed_history = defaultdict(
    lambda: deque(maxlen=10)
)

seen_ids = {
    "Person": set(),
    "Car": set(),
    "Cat": set()
}

# Save a few frames for Jupyter display
sample_frames = []

sample_positions = np.linspace(
    0,
    max(total_frames - 1, 0),
    10
).astype(int)

sample_set = set(sample_positions)

# ============================================================
# PROCESS VIDEO
# ============================================================

frame_number = 0

print()
print("PROCESSING VIDEO...")
print()

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # --------------------------------------------------------
    # YOLO + BoT-SORT
    # --------------------------------------------------------

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        classes=[0, 2, 15],
        conf=CONFIDENCE,
        iou=0.5,
        verbose=False
    )

    result = results[0]

    # --------------------------------------------------------
    # DETECTIONS
    # --------------------------------------------------------

    if (
        result.boxes is not None
        and result.boxes.id is not None
    ):

        boxes = result.boxes.xyxy.cpu().numpy()

        ids = (
            result.boxes.id
            .cpu()
            .numpy()
            .astype(int)
        )

        classes = (
            result.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        confidences = (
            result.boxes.conf
            .cpu()
            .numpy()
        )

        for box, track_id, cls, confidence in zip(
            boxes,
            ids,
            classes,
            confidences
        ):

            if cls not in TARGET_CLASSES:
                continue

            object_name = TARGET_CLASSES[cls]

            x1, y1, x2, y2 = map(
                int,
                box
            )

            # ------------------------------------------------
            # Ground position
            # ------------------------------------------------

            center_x = int(
                (x1 + x2) / 2
            )

            bottom_y = int(y2)

            current_position = np.array(
                [center_x, bottom_y],
                dtype=np.float32
            )

            # ------------------------------------------------
            # Track ID
            # ------------------------------------------------

            seen_ids[
                object_name
            ].add(track_id)

            # ------------------------------------------------
            # Pixel speed
            # ------------------------------------------------

            pixel_speed = 0.0

            if track_id in previous_positions:

                previous_position = (
                    previous_positions[track_id]
                )

                distance = np.linalg.norm(
                    current_position
                    - previous_position
                )

                pixel_speed = distance * fps

            previous_positions[
                track_id
            ] = current_position

            # ------------------------------------------------
            # Smooth speed
            # ------------------------------------------------

            speed_history[
                track_id
            ].append(pixel_speed)

            smooth_speed = np.mean(
                speed_history[track_id]
            )

            # ------------------------------------------------
            # Trail
            # ------------------------------------------------

            trails[
                track_id
            ].append(
                (center_x, bottom_y)
            )

            # ------------------------------------------------
            # Bounding box
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )

            # ------------------------------------------------
            # Ground point
            # ------------------------------------------------

            cv2.circle(
                frame,
                (center_x, bottom_y),
                5,
                (255, 255, 255),
                -1
            )

            # ------------------------------------------------
            # Trail
            # ------------------------------------------------

            points = list(
                trails[track_id]
            )

            for i in range(
                1,
                len(points)
            ):

                cv2.line(
                    frame,
                    points[i - 1],
                    points[i],
                    (255, 255, 255),
                    2
                )

            # ------------------------------------------------
            # Labels
            # ------------------------------------------------

            label = (
                f"{object_name} "
                f"ID:{track_id}"
            )

            speed_label = (
                f"{smooth_speed:.1f} px/s"
            )

            # ------------------------------------------------
            # Background
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (x1, max(0, y1 - 50)),
                (x1 + 190, y1),
                (0, 0, 0),
                -1
            )

            # ------------------------------------------------
            # Object label
            # ------------------------------------------------

            cv2.putText(
                frame,
                label,
                (x1 + 5, max(20, y1 - 28)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )

            # ------------------------------------------------
            # Speed label
            # ------------------------------------------------

            cv2.putText(
                frame,
                speed_label,
                (x1 + 5, max(40, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.50,
                (255, 255, 255),
                2
            )

    # ========================================================
    # FRAME COUNTER
    # ========================================================

    cv2.putText(
        frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    # ========================================================
    # SAVE SAMPLE FRAMES
    # ========================================================

    if frame_number - 1 in sample_set:

        sample_frames.append(
            cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
        )

    # ========================================================
    # WRITE FRAME
    # ========================================================

    out.write(frame)

    # Progress
    if frame_number % 100 == 0:

        progress = (
            frame_number
            / total_frames
            * 100
        )

        print(
            f"Processed: "
            f"{frame_number}/{total_frames} "
            f"({progress:.1f}%)"
        )

# ============================================================
# RELEASE
# ============================================================

cap.release()
out.release()

print()
print("PROCESSING FINISHED")
print("-------------------")
print(f"Output: {OUTPUT_PATH}")

# ============================================================
# TRACKING SUMMARY
# ============================================================

print()
print("TRACKING SUMMARY")
print("----------------")

for object_name in [
    "Person",
    "Car",
    "Cat"
]:

    print(
        f"{object_name}: "
        f"{len(seen_ids[object_name])} "
        f"unique IDs"
    )

# ============================================================
# DISPLAY 10 SAMPLE FRAMES
# ============================================================

if len(sample_frames) > 0:

    rows = 5
    cols = 2

    plt.figure(
        figsize=(16, 20)
    )

    for i, image in enumerate(
        sample_frames
    ):

        plt.subplot(
            rows,
            cols,
            i + 1
        )

        plt.imshow(image)

        plt.axis("off")

        plt.title(
            f"Sample frame {i + 1}"
        )

    plt.tight_layout()

    plt.show()

In [ ]:
import cv2
import re
import math
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict, deque
from ultralytics import YOLO
from ultralytics.trackers.utils.gmc import GMC
import easyocr


# ============================================================
# SETTINGS
# ============================================================

VIDEO_PATH = r"\\192.168.1.10\shear all\test_vedio\VID00054.AVI"

OUTPUT_PATH = r"C:\Users\PC-LAB1\Desktop\shok\notebooks\Day_08\drone_tracking_result_4.mp4"

MODEL_PATH = "yolo11n.pt"


# ============================================================
# CAMERA SETTINGS
# ============================================================

# IMPORTANT:
# Replace this with the horizontal FOV of your drone camera
# if you know it.
#
# 80 degrees is only an initial estimate.
CAMERA_HFOV_DEG = 80.0

# True if the camera is pointing approximately straight down.
CAMERA_IS_NADIR = True


# ============================================================
# DETECTION SETTINGS
# ============================================================

CONFIDENCE = 0.25

# COCO classes:
# 0  = person
# 2  = car
# 15 = cat

TARGET_CLASSES = {
    0: "Person",
    2: "Car",
    15: "Cat"
}


# ============================================================
# OCR SETTINGS
# ============================================================

# ALT text is in the top-right/right-center.

ALT_ROI_X = 0.70
ALT_ROI_Y = 0.00
ALT_ROI_W = 0.30
ALT_ROI_H = 0.20

# OCR every 5 frames.
# At 25 FPS = about 5 OCR operations/sec.
OCR_EVERY_N_FRAMES = 5

# Keep the previous altitude if OCR temporarily fails.
ALT_HOLD_FRAMES = 75


# ============================================================
# SPEED SETTINGS
# ============================================================

# Smooth speed over several frames.
SPEED_SMOOTHING = 10

# Ignore unrealistic tracking jumps.
MAX_SPEED_KMH = 200.0


# ============================================================
# CAMERA MOTION COMPENSATION
# ============================================================

# Ultralytics GMC method.
GMC_METHOD = "sparseOptFlow"

GMC_DOWNSCALE = 2

# Ignore the ALT/HUD area when calculating camera motion.
GMC_MASK_X = 0.68
GMC_MASK_Y = 0.00
GMC_MASK_W = 0.32
GMC_MASK_H = 0.20


# ============================================================
# DISPLAY
# ============================================================

NUM_SAMPLE_FRAMES = 10

SHOW_PROGRESS_EVERY = 100


# ============================================================
# CHECK VIDEO
# ============================================================

import os

print("Checking video...")

if not os.path.exists(VIDEO_PATH):

    raise FileNotFoundError(
        f"\nVideo was not found:\n{VIDEO_PATH}\n\n"
        "Check that the network path is accessible from Windows."
    )

print("Video found.")
print()


# ============================================================
# LOAD YOLO
# ============================================================

print("Loading YOLO model...")

model = YOLO(MODEL_PATH)

print("YOLO loaded.")
print()


# ============================================================
# LOAD OCR
# ============================================================

print("Loading EasyOCR...")

reader = easyocr.Reader(
    ["en"],
    gpu=False
)

print("EasyOCR loaded.")
print()


# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():

    raise RuntimeError(
        f"Could not open video:\n{VIDEO_PATH}"
    )


fps = cap.get(
    cv2.CAP_PROP_FPS
)

if fps <= 0:

    fps = 25.0


width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)

duration = (
    total_frames / fps
    if fps > 0
    else 0
)


print("VIDEO INFORMATION")
print("-----------------")
print(f"Resolution : {width} x {height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")
print(f"Duration   : {duration:.2f} seconds")
print()


# ============================================================
# OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():

    raise RuntimeError(
        f"Could not create output video:\n{OUTPUT_PATH}"
    )


# ============================================================
# GLOBAL MOTION COMPENSATION
# ============================================================

print("Initializing camera-motion compensation...")

gmc = GMC(
    method=GMC_METHOD,
    downscale=GMC_DOWNSCALE
)

print("GMC ready.")
print()


# ============================================================
# TRACKING VARIABLES
# ============================================================

previous_positions = {}

speed_history = defaultdict(
    lambda: deque(
        maxlen=SPEED_SMOOTHING
    )
)

seen_ids = {
    "Person": set(),
    "Car": set(),
    "Cat": set()
}


# ============================================================
# ALTITUDE VARIABLES
# ============================================================

last_altitude = None

last_altitude_frame = -999999

altitude_history = deque(
    maxlen=15
)


# ============================================================
# SAMPLE FRAMES
# ============================================================

sample_frames = []

sample_indices = set(
    np.linspace(
        0,
        max(total_frames - 1, 0),
        NUM_SAMPLE_FRAMES
    ).astype(int)
)


# ============================================================
# FUNCTIONS
# ============================================================

def get_alt_roi(frame):

    h, w = frame.shape[:2]

    x1 = int(
        w * ALT_ROI_X
    )

    y1 = int(
        h * ALT_ROI_Y
    )

    x2 = int(
        w * (
            ALT_ROI_X
            + ALT_ROI_W
        )
    )

    y2 = int(
        h * (
            ALT_ROI_Y
            + ALT_ROI_H
        )
    )

    x1 = max(
        0,
        min(x1, w - 1)
    )

    y1 = max(
        0,
        min(y1, h - 1)
    )

    x2 = max(
        x1 + 1,
        min(x2, w)
    )

    y2 = max(
        y1 + 1,
        min(y2, h)
    )

    return (
        frame[y1:y2, x1:x2],
        (x1, y1, x2, y2)
    )


def read_altitude(frame):

    roi, _ = get_alt_roi(frame)

    # Enlarge the ALT region.
    roi = cv2.resize(
        roi,
        None,
        fx=3.0,
        fy=3.0,
        interpolation=cv2.INTER_CUBIC
    )

    # Grayscale.
    gray = cv2.cvtColor(
        roi,
        cv2.COLOR_BGR2GRAY
    )

    # Improve contrast.
    gray = cv2.convertScaleAbs(
        gray,
        alpha=1.7,
        beta=0
    )

    # Slight sharpening.
    blur = cv2.GaussianBlur(
        gray,
        (3, 3),
        0
    )

    enhanced = cv2.addWeighted(
        gray,
        1.5,
        blur,
        -0.5,
        0
    )

    results = reader.readtext(
        enhanced,
        detail=1,
        paragraph=False
    )

    altitude_values = []

    for result in results:

        if len(result) < 2:
            continue

        text = str(
            result[1]
        ).upper()

        # Common OCR corrections.
        text = text.replace(
            "O",
            "0"
        )

        text = text.replace(
            "I",
            "1"
        )

        text = text.replace(
            "L",
            "1"
        )

        # ----------------------------------------------------
        # Look specifically for:
        #
        # ALT 52 M
        # ALT: 52m
        # ALT 52.5 M
        # ----------------------------------------------------

        matches = re.findall(
            r"ALT\s*[^0-9]{0,8}"
            r"([0-9]+(?:\.[0-9]+)?)"
            r"\s*M?",
            text
        )

        for match in matches:

            try:

                value = float(match)

                if (
                    0.5
                    <= value
                    <= 1000
                ):

                    altitude_values.append(
                        value
                    )

            except:
                pass

        # ----------------------------------------------------
        # If OCR missed "ALT", look for number + M.
        # ----------------------------------------------------

        if len(altitude_values) == 0:

            matches = re.findall(
                r"([0-9]+(?:\.[0-9]+)?)"
                r"\s*M",
                text
            )

            for match in matches:

                try:

                    value = float(match)

                    if (
                        0.5
                        <= value
                        <= 1000
                    ):

                        altitude_values.append(
                            value
                        )

                except:
                    pass

    if len(altitude_values) == 0:

        return None

    # Median makes OCR more stable.
    altitude = float(
        np.median(
            altitude_values
        )
    )

    return altitude


def estimate_meter_per_pixel(
    altitude,
    image_width
):

    if altitude is None:
        return None

    hfov_rad = math.radians(
        CAMERA_HFOV_DEG
    )

    # Approximate ground width.
    ground_width = (
        2.0
        * altitude
        * math.tan(
            hfov_rad / 2.0
        )
    )

    meter_per_pixel = (
        ground_width
        / image_width
    )

    return meter_per_pixel


def transform_point(
    point,
    warp
):

    x = float(point[0])
    y = float(point[1])

    new_x = (
        warp[0, 0] * x
        + warp[0, 1] * y
        + warp[0, 2]
    )

    new_y = (
        warp[1, 0] * x
        + warp[1, 1] * y
        + warp[1, 2]
    )

    return np.array(
        [new_x, new_y],
        dtype=np.float32
    )


def create_gmc_frame(frame):

    gmc_frame = frame.copy()

    h, w = frame.shape[:2]

    x1 = int(
        w * GMC_MASK_X
    )

    y1 = int(
        h * GMC_MASK_Y
    )

    x2 = int(
        w * (
            GMC_MASK_X
            + GMC_MASK_W
        )
    )

    y2 = int(
        h * (
            GMC_MASK_Y
            + GMC_MASK_H
        )
    )

    # Black out ALT/HUD.
    cv2.rectangle(
        gmc_frame,
        (x1, y1),
        (x2, y2),
        (0, 0, 0),
        -1
    )

    return gmc_frame


# ============================================================
# PROCESS VIDEO
# ============================================================

frame_number = 0

print("PROCESSING VIDEO")
print("================")
print()


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1


    # ========================================================
    # READ ALTITUDE
    # ========================================================

    if (
        frame_number == 1
        or frame_number
        % OCR_EVERY_N_FRAMES == 0
    ):

        detected_altitude = (
            read_altitude(frame)
        )

        if detected_altitude is not None:

            last_altitude = (
                detected_altitude
            )

            last_altitude_frame = (
                frame_number
            )

            altitude_history.append(
                detected_altitude
            )


    # Keep last altitude temporarily.
    if (
        frame_number
        - last_altitude_frame
        <= ALT_HOLD_FRAMES
    ):

        current_altitude = (
            last_altitude
        )

    else:

        current_altitude = None


    # ========================================================
    # CAMERA MOTION
    # ========================================================

    gmc_frame = create_gmc_frame(
        frame
    )

    warp = gmc.apply(
        gmc_frame
    )


    # ========================================================
    # YOLO TRACKING
    # ========================================================

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        classes=[
            0,
            2,
            15
        ],
        conf=CONFIDENCE,
        iou=0.5,
        verbose=False
    )

    result = results[0]


    # ========================================================
    # PROCESS DETECTIONS
    # ========================================================

    if (
        result.boxes is not None
        and result.boxes.id is not None
    ):

        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        ids = (
            result.boxes.id
            .cpu()
            .numpy()
            .astype(int)
        )

        classes = (
            result.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        confidences = (
            result.boxes.conf
            .cpu()
            .numpy()
        )


        for (
            box,
            track_id,
            cls,
            confidence
        ) in zip(
            boxes,
            ids,
            classes,
            confidences
        ):

            if cls not in TARGET_CLASSES:
                continue


            object_name = (
                TARGET_CLASSES[cls]
            )


            x1, y1, x2, y2 = map(
                int,
                box
            )


            # ------------------------------------------------
            # Bottom center
            # ------------------------------------------------

            current_position = np.array(
                [
                    (x1 + x2) / 2.0,
                    float(y2)
                ],
                dtype=np.float32
            )


            # ------------------------------------------------
            # Save ID
            # ------------------------------------------------

            seen_ids[
                object_name
            ].add(
                track_id
            )


            # ------------------------------------------------
            # SPEED
            # ------------------------------------------------

            speed_kmh = 0.0


            if (
                track_id
                in previous_positions
            ):

                previous_position = (
                    previous_positions[
                        track_id
                    ]
                )


                # Camera compensation:
                #
                # Transform the old point into
                # the current frame.
                compensated_previous = (
                    transform_point(
                        previous_position,
                        warp
                    )
                )


                # Remaining movement should
                # be closer to object movement.
                movement_pixels = (
                    np.linalg.norm(
                        current_position
                        - compensated_previous
                    )
                )


                # ------------------------------------------------
                # Pixel -> meters
                # ------------------------------------------------

                meter_per_pixel = (
                    estimate_meter_per_pixel(
                        current_altitude,
                        width
                    )
                )


                if (
                    meter_per_pixel
                    is not None
                ):

                    distance_m = (
                        movement_pixels
                        * meter_per_pixel
                    )


                    # meters/sec
                    speed_ms = (
                        distance_m
                        * fps
                    )


                    # km/h
                    raw_speed_kmh = (
                        speed_ms
                        * 3.6
                    )


                    # Ignore impossible tracking jumps.
                    if (
                        0
                        <= raw_speed_kmh
                        <= MAX_SPEED_KMH
                    ):

                        speed_history[
                            track_id
                        ].append(
                            raw_speed_kmh
                        )


            # ------------------------------------------------
            # Save current position
            # ------------------------------------------------

            previous_positions[
                track_id
            ] = current_position


            # ------------------------------------------------
            # Smooth speed
            # ------------------------------------------------

            if len(
                speed_history[
                    track_id
                ]
            ) > 0:

                speed_kmh = float(
                    np.median(
                        speed_history[
                            track_id
                        ]
                    )
                )


            # =================================================
            # DRAW BOX
            # =================================================

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )


            # =================================================
            # LABEL
            # =================================================

            label = (
                f"{object_name} "
                f"ID:{track_id}"
            )

            speed_text = (
                f"{speed_kmh:.1f} km/h"
            )


            # ------------------------------------------------
            # Label size
            # ------------------------------------------------

            (label_w, label_h), _ = (
                cv2.getTextSize(
                    label,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    2
                )
            )

            (speed_w, speed_h), _ = (
                cv2.getTextSize(
                    speed_text,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    2
                )
            )


            text_width = max(
                label_w,
                speed_w
            ) + 10


            # ------------------------------------------------
            # Black label background
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (
                    x1,
                    max(
                        0,
                        y1 - 52
                    )
                ),
                (
                    x1 + text_width,
                    y1
                ),
                (0, 0, 0),
                -1
            )


            # ------------------------------------------------
            # Object label
            # ------------------------------------------------

            cv2.putText(
                frame,
                label,
                (
                    x1 + 5,
                    max(
                        20,
                        y1 - 30
                    )
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )


            # ------------------------------------------------
            # Speed
            # ------------------------------------------------

            cv2.putText(
                frame,
                speed_text,
                (
                    x1 + 5,
                    max(
                        42,
                        y1 - 8
                    )
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )


    # ========================================================
    # ALTITUDE DISPLAY
    # ========================================================

    if current_altitude is not None:

        altitude_text = (
            f"ALT: "
            f"{current_altitude:.1f} m"
        )

    else:

        altitude_text = (
            "ALT: --"
        )


    cv2.putText(
        frame,
        altitude_text,
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    # ========================================================
    # FRAME COUNTER
    # ========================================================

    cv2.putText(
        frame,
        f"Frame: "
        f"{frame_number}/"
        f"{total_frames}",
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    # ========================================================
    # SAVE SAMPLE FRAME
    # ========================================================

    if (
        frame_number - 1
        in sample_indices
    ):

        sample_frames.append(
            cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
        )


    # ========================================================
    # WRITE OUTPUT
    # ========================================================

    out.write(frame)


    # ========================================================
    # PROGRESS
    # ========================================================

    if (
        frame_number
        % SHOW_PROGRESS_EVERY
        == 0
    ):

        progress = (
            frame_number
            / total_frames
            * 100
        )

        print(
            f"Processed "
            f"{frame_number}/"
            f"{total_frames} "
            f"({progress:.1f}%)"
        )


# ============================================================
# RELEASE
# ============================================================

cap.release()
out.release()


# ============================================================
# RESULTS
# ============================================================

print()
print("===================================")
print("PROCESSING FINISHED")
print("===================================")

print(
    f"Output video:\n{OUTPUT_PATH}"
)

print()


# ============================================================
# TRACKING SUMMARY
# ============================================================

print("TRACKING SUMMARY")
print("----------------")

for object_name in [
    "Person",
    "Car",
    "Cat"
]:

    print(
        f"{object_name}: "
        f"{len(seen_ids[object_name])} "
        f"unique IDs"
    )


# ============================================================
# ALTITUDE SUMMARY
# ============================================================

print()
print("ALTITUDE SUMMARY")
print("----------------")

if len(altitude_history) > 0:

    print(
        f"Minimum recent ALT: "
        f"{min(altitude_history):.1f} m"
    )

    print(
        f"Maximum recent ALT: "
        f"{max(altitude_history):.1f} m"
    )

    print(
        f"Average recent ALT: "
        f"{np.mean(altitude_history):.1f} m"
    )

else:

    print(
        "No altitude was detected."
    )

    print(
        "Check ALT_ROI_X/Y/W/H."
    )


# ============================================================
# DISPLAY SAMPLE FRAMES
# ============================================================

if len(sample_frames) > 0:

    rows = math.ceil(
        len(sample_frames) / 2
    )

    cols = 2

    plt.figure(
        figsize=(16, 5 * rows)
    )

    for i, image in enumerate(
        sample_frames
    ):

        plt.subplot(
            rows,
            cols,
            i + 1
        )

        plt.imshow(image)

        plt.axis("off")

        plt.title(
            f"Sample frame {i + 1}"
        )

    plt.tight_layout()

    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os


# ============================================================
# SETTINGS
# ============================================================

VIDEO_PATH = r"C:\Users\PC-LAB1\Desktop\New folder\Test Video.mp4"

OUTPUT_PATH = r"C:\Users\PC-LAB1\Desktop\shok\notebooks\Day_08\drone_tracking_result_test.mp4"

MODEL_PATH = "yolo11n.pt"

CONFIDENCE = 0.25

# COCO classes
# 0  = Person
# 2  = Car
# 15 = Cat

TARGET_CLASSES = {
    0: "Person",
    2: "Car",
    15: "Cat"
}

NUM_SAMPLE_FRAMES = 10

SHOW_PROGRESS_EVERY = 100


# ============================================================
# CHECK VIDEO
# ============================================================

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"Video not found:\n{VIDEO_PATH}"
    )

print("Video found.")


# ============================================================
# LOAD YOLO
# ============================================================

print("Loading YOLO model...")

model = YOLO(MODEL_PATH)

print("YOLO loaded.")


# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video:\n{VIDEO_PATH}"
    )


fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 25.0


width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

duration = total_frames / fps


print()
print("VIDEO INFORMATION")
print("------------------")
print(f"Resolution : {width} x {height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")
print(f"Duration   : {duration:.2f} seconds")


# ============================================================
# OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():
    raise RuntimeError(
        f"Could not create output:\n{OUTPUT_PATH}"
    )


# ============================================================
# TRACKING INFORMATION
# ============================================================

seen_ids = {
    "Person": set(),
    "Car": set(),
    "Cat": set()
}


# ============================================================
# SAMPLE FRAMES
# ============================================================

sample_frames = []

sample_indices = set(
    np.linspace(
        0,
        max(total_frames - 1, 0),
        NUM_SAMPLE_FRAMES
    ).astype(int)
)


# ============================================================
# PROCESS VIDEO
# ============================================================

frame_number = 0

print()
print("PROCESSING VIDEO...")
print()


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1


    # ========================================================
    # YOLO + BOT-SORT TRACKING
    # ========================================================

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        classes=[
            0,   # Person
            2,   # Car
            15   # Cat
        ],
        conf=CONFIDENCE,
        iou=0.5,
        verbose=False
    )

    result = results[0]


    # ========================================================
    # PROCESS DETECTIONS
    # ========================================================

    if (
        result.boxes is not None
        and result.boxes.id is not None
    ):

        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        ids = (
            result.boxes.id
            .cpu()
            .numpy()
            .astype(int)
        )

        classes = (
            result.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        confidences = (
            result.boxes.conf
            .cpu()
            .numpy()
        )


        for (
            box,
            track_id,
            cls,
            confidence
        ) in zip(
            boxes,
            ids,
            classes,
            confidences
        ):

            if cls not in TARGET_CLASSES:
                continue


            object_name = TARGET_CLASSES[cls]


            x1, y1, x2, y2 = map(
                int,
                box
            )


            # ------------------------------------------------
            # Save tracking ID
            # ------------------------------------------------

            seen_ids[
                object_name
            ].add(track_id)


            # ------------------------------------------------
            # Bounding box
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )


            # ------------------------------------------------
            # Label
            # ------------------------------------------------

            label = (
                f"{object_name} "
                f"ID:{track_id}"
            )


            # ------------------------------------------------
            # Label size
            # ------------------------------------------------

            (
                text_width,
                text_height
            ), baseline = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )


            # ------------------------------------------------
            # Label background
            # ------------------------------------------------

            label_y1 = max(
                0,
                y1 - text_height - 10
            )

            label_y2 = y1


            cv2.rectangle(
                frame,
                (
                    x1,
                    label_y1
                ),
                (
                    x1 + text_width + 10,
                    label_y2
                ),
                (0, 0, 0),
                -1
            )


            # ------------------------------------------------
            # Label text
            # ------------------------------------------------

            cv2.putText(
                frame,
                label,
                (
                    x1 + 5,
                    y1 - 7
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )


    # ========================================================
    # FRAME NUMBER
    # ========================================================

    cv2.putText(
        frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2
    )


    # ========================================================
    # SAVE SAMPLE FRAMES
    # ========================================================

    if frame_number - 1 in sample_indices:

        sample_frames.append(
            cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
        )


    # ========================================================
    # WRITE FRAME
    # ========================================================

    out.write(frame)


    # ========================================================
    # PROGRESS
    # ========================================================

    if (
        frame_number
        % SHOW_PROGRESS_EVERY
        == 0
    ):

        progress = (
            frame_number
            / total_frames
            * 100
        )

        print(
            f"Processed "
            f"{frame_number}/"
            f"{total_frames} "
            f"({progress:.1f}%)"
        )


# ============================================================
# RELEASE
# ============================================================

cap.release()
out.release()


# ============================================================
# SUMMARY
# ============================================================

print()
print("===================================")
print("PROCESSING FINISHED")
print("===================================")

print()
print("Output:")
print(OUTPUT_PATH)

print()
print("TRACKING SUMMARY")
print("----------------")

for object_name in [
    "Person",
    "Car",
    "Cat"
]:

    print(
        f"{object_name}: "
        f"{len(seen_ids[object_name])} "
        f"unique IDs"
    )


# ============================================================
# DISPLAY SAMPLE FRAMES
# ============================================================

if len(sample_frames) > 0:

    rows = int(
        np.ceil(
            len(sample_frames) / 2
        )
    )

    plt.figure(
        figsize=(16, 5 * rows)
    )

    for i, image in enumerate(
        sample_frames
    ):

        plt.subplot(
            rows,
            2,
            i + 1
        )

        plt.imshow(image)

        plt.axis("off")

        plt.title(
            f"Sample Frame {i + 1}"
        )

    plt.tight_layout()

    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os


# ============================================================
# SETTINGS
# ============================================================

VIDEO_PATH = r"C:\Users\PC-LAB1\Desktop\New folder\Test Video.mp4"

OUTPUT_PATH = r"C:\Users\PC-LAB1\Desktop\shok\notebooks\Day_08\road_tracking_result_test_2.mp4"

MODEL_PATH = "yolo11s.pt"

# Lower confidence helps detect distant/small cars
CONFIDENCE = 0.15

# Larger input resolution helps small objects
IMAGE_SIZE = 1280

# COCO classes
# 0  = Person
# 2  = Car
# 15 = Cat

TARGET_CLASSES = {
    0: "Person",
    2: "Car",
    15: "Cat"
}

NUM_SAMPLE_FRAMES = 10

SHOW_PROGRESS_EVERY = 100


# ============================================================
# CHECK VIDEO
# ============================================================

if not os.path.exists(VIDEO_PATH):

    raise FileNotFoundError(
        f"Video not found:\n{VIDEO_PATH}"
    )

print("Video found.")


# ============================================================
# LOAD YOLO
# ============================================================

print("Loading YOLO11s...")

model = YOLO(MODEL_PATH)

print("YOLO11s loaded.")


# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

if not cap.isOpened():

    raise RuntimeError(
        f"Could not open video:\n{VIDEO_PATH}"
    )


fps = cap.get(
    cv2.CAP_PROP_FPS
)

if fps <= 0:
    fps = 25.0


width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)

duration = (
    total_frames / fps
)


print()
print("VIDEO INFORMATION")
print("------------------")
print(f"Resolution : {width} x {height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")
print(f"Duration   : {duration:.2f} seconds")
print()


# ============================================================
# OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():

    raise RuntimeError(
        f"Could not create output video:\n{OUTPUT_PATH}"
    )


# ============================================================
# TRACKING SUMMARY
# ============================================================

seen_ids = {
    "Person": set(),
    "Car": set(),
    "Cat": set()
}


# ============================================================
# SAMPLE FRAMES
# ============================================================

sample_frames = []

sample_indices = set(
    np.linspace(
        0,
        max(total_frames - 1, 0),
        NUM_SAMPLE_FRAMES
    ).astype(int)
)


# ============================================================
# PROCESS VIDEO
# ============================================================

frame_number = 0

print("PROCESSING VIDEO...")
print()


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1


    # ========================================================
    # YOLO11s + BOT-SORT
    # ========================================================

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",

        # Person, car, cat
        classes=[
            0,
            2,
            15
        ],

        # Lower confidence for small/far objects
        conf=CONFIDENCE,

        # NMS IoU
        iou=0.5,

        # Larger inference resolution
        imgsz=IMAGE_SIZE,

        verbose=False
    )

    result = results[0]


    # ========================================================
    # PROCESS DETECTIONS
    # ========================================================

    if (
        result.boxes is not None
        and result.boxes.id is not None
    ):

        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        ids = (
            result.boxes.id
            .cpu()
            .numpy()
            .astype(int)
        )

        classes = (
            result.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        confidences = (
            result.boxes.conf
            .cpu()
            .numpy()
        )


        for (
            box,
            track_id,
            cls,
            confidence
        ) in zip(
            boxes,
            ids,
            classes,
            confidences
        ):

            if cls not in TARGET_CLASSES:
                continue


            object_name = (
                TARGET_CLASSES[cls]
            )


            x1, y1, x2, y2 = map(
                int,
                box
            )


            # ------------------------------------------------
            # SAVE TRACKING ID
            # ------------------------------------------------

            seen_ids[
                object_name
            ].add(
                track_id
            )


            # ------------------------------------------------
            # BOUNDING BOX
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )


            # ------------------------------------------------
            # LABEL
            # ------------------------------------------------

            label = (
                f"{object_name} "
                f"ID:{track_id}"
            )


            # ------------------------------------------------
            # LABEL SIZE
            # ------------------------------------------------

            (
                text_width,
                text_height
            ), baseline = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )


            # ------------------------------------------------
            # LABEL BACKGROUND
            # ------------------------------------------------

            label_y1 = max(
                0,
                y1 - text_height - 10
            )

            label_y2 = y1


            cv2.rectangle(
                frame,
                (
                    x1,
                    label_y1
                ),
                (
                    x1 + text_width + 10,
                    label_y2
                ),
                (0, 0, 0),
                -1
            )


            # ------------------------------------------------
            # LABEL
            # ------------------------------------------------

            cv2.putText(
                frame,
                label,
                (
                    x1 + 5,
                    y1 - 7
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )


    # ========================================================
    # FRAME NUMBER
    # ========================================================

    cv2.putText(
        frame,
        f"Frame: {frame_number}/{total_frames}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2
    )


    # ========================================================
    # SAVE SAMPLE FRAME
    # ========================================================

    if (
        frame_number - 1
        in sample_indices
    ):

        sample_frames.append(
            cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
        )


    # ========================================================
    # WRITE VIDEO
    # ========================================================

    out.write(
        frame
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (
        frame_number
        % SHOW_PROGRESS_EVERY
        == 0
    ):

        progress = (
            frame_number
            / total_frames
            * 100
        )

        print(
            f"Processed "
            f"{frame_number}/"
            f"{total_frames} "
            f"({progress:.1f}%)"
        )


# ============================================================
# RELEASE
# ============================================================

cap.release()

out.release()


# ============================================================
# SUMMARY
# ============================================================

print()
print("===================================")
print("PROCESSING FINISHED")
print("===================================")

print()
print("Output video:")
print(OUTPUT_PATH)

print()
print("TRACKING SUMMARY")
print("----------------")

for object_name in [
    "Person",
    "Car",
    "Cat"
]:

    print(
        f"{object_name}: "
        f"{len(seen_ids[object_name])} "
        f"unique IDs"
    )


# ============================================================
# DISPLAY SAMPLE FRAMES
# ============================================================

if len(sample_frames) > 0:

    rows = int(
        np.ceil(
            len(sample_frames) / 2
        )
    )

    plt.figure(
        figsize=(16, 5 * rows)
    )

    for i, image in enumerate(
        sample_frames
    ):

        plt.subplot(
            rows,
            2,
            i + 1
        )

        plt.imshow(
            image
        )

        plt.axis(
            "off"
        )

        plt.title(
            f"Sample Frame {i + 1}"
        )

    plt.tight_layout()

    plt.show()

In [ ]:
import cv2
import numpy as np
import os
from collections import defaultdict, deque
from ultralytics import YOLO


# ============================================================
# SETTINGS
# ============================================================

VIDEO_PATH = r"C:\Users\PC-LAB1\Desktop\New folder\Test Video.mp4"

OUTPUT_PATH = r"C:\Users\PC-LAB1\Desktop\shok\notebooks\Day_08\road_tracking_speed.mp4"

MODEL_PATH = "yolo11s.pt"

CONFIDENCE = 0.15
IMAGE_SIZE = 1280


# ============================================================
# OBJECT CLASS
# ============================================================

# COCO class:
# 2 = car

CAR_CLASS = 2


# ============================================================
# APPROXIMATE ROAD CALIBRATION
# ============================================================
#
# Based on the 1280x720 video frame you showed.
#
# The selected area represents approximately:
#
#   Road width  ≈ 7 meters
#   Road depth  ≈ 50 meters
#
# These are APPROXIMATE values.
#
# P1 ---------------- P2
#  \                  /
#   \                /
#    \              /
#     \            /
#      P4 -------- P3
#

IMAGE_POINTS = np.float32([
    [425, 275],      # P1 - far left
    [520, 275],      # P2 - far right
    [595, 720],      # P3 - near right
    [145, 720]       # P4 - near left
])


ROAD_WIDTH_METERS = 7.0
ROAD_LENGTH_METERS = 50.0


# ============================================================
# BIRD'S-EYE TRANSFORMATION SIZE
# ============================================================

WARP_WIDTH = 700
WARP_HEIGHT = 5000


# ============================================================
# SPEED SETTINGS
# ============================================================

# Median of the last N speed measurements
SPEED_SMOOTHING = 8

# Ignore impossible values
MAX_SPEED_KMH = 150.0

# Ignore tiny movements caused by tracking jitter
MIN_MOVEMENT_METERS = 0.05


# ============================================================
# CHECK VIDEO
# ============================================================

if not os.path.exists(VIDEO_PATH):

    raise FileNotFoundError(
        f"Video not found:\n{VIDEO_PATH}"
    )

print("Video found.")


# ============================================================
# LOAD YOLO11s
# ============================================================

print("Loading YOLO11s...")

model = YOLO(MODEL_PATH)

print("YOLO11s loaded.")


# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():

    raise RuntimeError(
        f"Could not open video:\n{VIDEO_PATH}"
    )


fps = cap.get(
    cv2.CAP_PROP_FPS
)

if fps <= 0:
    fps = 25.0


width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


print()
print("VIDEO INFORMATION")
print("------------------")
print(f"Resolution : {width} x {height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")
print(
    f"Duration   : "
    f"{total_frames / fps:.2f} seconds"
)


# ============================================================
# PERSPECTIVE TRANSFORMATION
# ============================================================

DESTINATION_POINTS = np.float32([
    [0, 0],
    [WARP_WIDTH, 0],
    [WARP_WIDTH, WARP_HEIGHT],
    [0, WARP_HEIGHT]
])


perspective_matrix = cv2.getPerspectiveTransform(
    IMAGE_POINTS,
    DESTINATION_POINTS
)


# ============================================================
# ROAD SCALE
# ============================================================

meters_per_pixel_x = (
    ROAD_WIDTH_METERS
    / WARP_WIDTH
)

meters_per_pixel_y = (
    ROAD_LENGTH_METERS
    / WARP_HEIGHT
)


# ============================================================
# IMAGE → ROAD COORDINATES
# ============================================================

def road_position(x, y):

    point = np.float32([
        [
            [x, y]
        ]
    ])

    transformed = cv2.perspectiveTransform(
        point,
        perspective_matrix
    )

    warped_x = transformed[0, 0, 0]
    warped_y = transformed[0, 0, 1]

    real_x = (
        warped_x
        * meters_per_pixel_x
    )

    real_y = (
        warped_y
        * meters_per_pixel_y
    )

    return np.array(
        [
            real_x,
            real_y
        ],
        dtype=np.float32
    )


# ============================================================
# CREATE OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():

    raise RuntimeError(
        f"Could not create output:\n{OUTPUT_PATH}"
    )


# ============================================================
# TRACKING DATA
# ============================================================

previous_positions = {}

last_seen_frame = {}

speed_history = defaultdict(
    lambda: deque(
        maxlen=SPEED_SMOOTHING
    )
)

seen_ids = set()


# ============================================================
# PROCESS VIDEO
# ============================================================

frame_number = 0

print()
print("PROCESSING VIDEO...")
print()


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1


    # ========================================================
    # YOLO11s + BOT-SORT
    # ========================================================

    results = model.track(
        frame,

        persist=True,

        tracker="botsort.yaml",

        classes=[
            CAR_CLASS
        ],

        conf=CONFIDENCE,

        iou=0.5,

        imgsz=IMAGE_SIZE,

        verbose=False
    )

    result = results[0]


    # ========================================================
    # PROCESS DETECTED CARS
    # ========================================================

    if (
        result.boxes is not None
        and result.boxes.id is not None
    ):

        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        ids = (
            result.boxes.id
            .cpu()
            .numpy()
            .astype(int)
        )


        for box, track_id in zip(
            boxes,
            ids
        ):

            track_id = int(track_id)


            # =================================================
            # BOUNDING BOX
            # =================================================

            x1, y1, x2, y2 = map(
                int,
                box
            )


            # =================================================
            # CAR ROAD-CONTACT POINT
            # =================================================
            #
            # Bottom-center of the bounding box.
            #
            # This is more appropriate for road-distance
            # calculations than the center of the box.
            #

            contact_x = (
                x1 + x2
            ) / 2.0

            contact_y = float(
                y2
            )


            # =================================================
            # CONVERT TO REAL-WORLD ROAD POSITION
            # =================================================

            current_position = road_position(
                contact_x,
                contact_y
            )


            speed_kmh = 0.0


            # =================================================
            # CALCULATE SPEED
            # =================================================

            if track_id in previous_positions:

                previous_position = (
                    previous_positions[
                        track_id
                    ]
                )

                previous_frame = (
                    last_seen_frame[
                        track_id
                    ]
                )


                frame_difference = (
                    frame_number
                    - previous_frame
                )


                if frame_difference > 0:

                    elapsed_time = (
                        frame_difference
                        / fps
                    )


                    distance_meters = (
                        np.linalg.norm(
                            current_position
                            - previous_position
                        )
                    )


                    # -----------------------------------------
                    # Ignore very small tracking movements
                    # -----------------------------------------

                    if (
                        distance_meters
                        >= MIN_MOVEMENT_METERS
                    ):

                        speed_ms = (
                            distance_meters
                            / elapsed_time
                        )


                        raw_speed_kmh = (
                            speed_ms
                            * 3.6
                        )


                        # -------------------------------------
                        # Reject impossible values
                        # -------------------------------------

                        if (
                            0
                            <= raw_speed_kmh
                            <= MAX_SPEED_KMH
                        ):

                            speed_history[
                                track_id
                            ].append(
                                raw_speed_kmh
                            )


            # =================================================
            # SAVE POSITION
            # =================================================

            previous_positions[
                track_id
            ] = current_position

            last_seen_frame[
                track_id
            ] = frame_number

            seen_ids.add(
                track_id
            )


            # =================================================
            # SMOOTH SPEED
            # =================================================

            if len(
                speed_history[
                    track_id
                ]
            ) > 0:

                speed_kmh = float(
                    np.median(
                        speed_history[
                            track_id
                        ]
                    )
                )


            # =================================================
            # DRAW BOUNDING BOX
            # =================================================

            cv2.rectangle(
                frame,

                (x1, y1),

                (x2, y2),

                (255, 255, 255),

                2
            )


            # =================================================
            # DRAW CONTACT POINT
            # =================================================

            cv2.circle(
                frame,

                (
                    int(contact_x),
                    int(contact_y)
                ),

                4,

                (255, 255, 255),

                -1
            )


            # =================================================
            # LABEL
            # =================================================

            id_text = (
                f"Car ID:{track_id}"
            )

            speed_text = (
                f"~{speed_kmh:.1f} km/h"
            )


            # =================================================
            # TEXT SIZE
            # =================================================

            (
                id_width,
                id_height
            ), _ = cv2.getTextSize(
                id_text,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )


            (
                speed_width,
                speed_height
            ), _ = cv2.getTextSize(
                speed_text,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )


            text_width = max(
                id_width,
                speed_width
            ) + 12


            text_height = (
                id_height
                + speed_height
                + 18
            )


            label_top = max(
                0,
                y1 - text_height
            )


            # =================================================
            # LABEL BACKGROUND
            # =================================================

            cv2.rectangle(
                frame,

                (
                    x1,
                    label_top
                ),

                (
                    x1 + text_width,
                    y1
                ),

                (0, 0, 0),

                -1
            )


            # =================================================
            # ID TEXT
            # =================================================

            cv2.putText(
                frame,

                id_text,

                (
                    x1 + 5,
                    y1 - speed_height - 8
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.55,

                (255, 255, 255),

                2
            )


            # =================================================
            # SPEED TEXT
            # =================================================

            cv2.putText(
                frame,

                speed_text,

                (
                    x1 + 5,
                    y1 - 5
                ),

                cv2.FONT_HERSHEY_SIMPLEX,

                0.55,

                (255, 255, 255),

                2
            )


    # ========================================================
    # DRAW CALIBRATION REGION
    # ========================================================

    calibration_points = (
        IMAGE_POINTS.astype(
            np.int32
        )
    )


    cv2.polylines(
        frame,

        [
            calibration_points
        ],

        True,

        (255, 255, 255),

        2
    )


    # ========================================================
    # FRAME NUMBER
    # ========================================================

    cv2.putText(
        frame,

        f"Frame: "
        f"{frame_number}/"
        f"{total_frames}",

        (20, 35),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.7,

        (255, 255, 255),

        2
    )


    # ========================================================
    # CALIBRATION INFORMATION
    # ========================================================

    cv2.putText(
        frame,

        "APPROX SPEED",

        (20, 65),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.55,

        (255, 255, 255),

        2
    )


    # ========================================================
    # WRITE FRAME
    # ========================================================

    out.write(
        frame
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if frame_number % 100 == 0:

        progress = (
            frame_number
            / total_frames
            * 100
        )

        print(
            f"Processed "
            f"{frame_number}/"
            f"{total_frames} "
            f"({progress:.1f}%)"
        )


# ============================================================
# RELEASE
# ============================================================

cap.release()

out.release()


# ============================================================
# SUMMARY
# ============================================================

print()
print("========================================")
print("PROCESSING FINISHED")
print("========================================")

print()
print(
    f"Unique cars tracked: "
    f"{len(seen_ids)}"
)

print()
print("Calibration:")
print(
    f"Approx road width  : "
    f"{ROAD_WIDTH_METERS} m"
)

print(
    f"Approx road length : "
    f"{ROAD_LENGTH_METERS} m"
)

print()
print("Output:")
print(
    OUTPUT_PATH
)

print()
print(
    "NOTE: Speeds are approximate."
)